In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama import ChatOllama
from langchain_community.vectorstores import FAISS
import os 
import string
import spacy 
import re
import contractions
from textblob import TextBlob
from langchain_text_splitters import RecursiveCharacterTextSplitter
from flashrank import Ranker,RerankRequest

C:\Users\Saravanan\AppData\Local\Temp\ipykernel_5160\4273577846.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
# pip install flashran

In [3]:
# pip show flashrank

### 1.Text Normalization

In [4]:
data = TextLoader('datas.txt').load()

In [5]:
for doc in data:
    doc.page_content = doc.page_content.lower()

In [6]:
data = (data[0].page_content)

In [7]:
print(type(data))

<class 'str'>


In [8]:
# import string
var = string.punctuation
data=re.sub(f'[{var}]',' ',data)
print(data)
data=re.sub(r'\s+',' ',data)
print(data)

  machine learning  deep learning  and natural language processing  nlp 

   1  introduction to artificial intelligence

artificial intelligence  ai  is one of the fastest growing technologies in the modern world  it is a branch of computer science that focuses on developing intelligent machines and computer systems capable of performing tasks that normally require human intelligence  these tasks include learning  problem solving  decision making  understanding languages  recognizing images  and analyzing information  artificial intelligence is widely used in healthcare  education  banking  transportation  business  and many other industries  some common examples of ai applications include chatgpt  google assistant  self driving cars  recommendation systems  facial recognition  and virtual assistants 

artificial intelligence consists of several important branches  including machine learning  deep learning  natural language processing  and computer vision  these technologies work toget

### 2.Lemma

In [9]:
nlp = spacy.load('en_core_web_sm') # converting the string to tokens
tokens = nlp(data)

In [10]:
updated_tokens = [str(token.lemma_) for token in tokens if not token.is_stop]
data = ' '.join(updated_tokens)
print(data)

  machine learn deep learning natural language processing nlp 1 introduction artificial intelligence artificial intelligence ai fast grow technology modern world branch computer science focus develop intelligent machine computer system capable perform task normally require human intelligence task include learn problem solve decision make understanding language recognize image analyze information artificial intelligence widely healthcare education banking transportation business industry common example ai application include chatgpt google assistant self drive car recommendation system facial recognition virtual assistant artificial intelligence consist important branch include machine learn deep learn natural language processing computer vision technology work machine intelligent capable solve real world problem 2 machine learning ml machine learning subset artificial intelligence enable computer learn historical datum identify pattern prediction decision explicitly program task instea

### 3.Chunking

In [11]:
chunksize = 150
chunks=[data[index:index+chunksize] for index in range(0,len(data),chunksize)]
chunks

['  machine learn deep learning natural language processing nlp 1 introduction artificial intelligence artificial intelligence ai fast grow technology m',
 'odern world branch computer science focus develop intelligent machine computer system capable perform task normally require human intelligence task in',
 'clude learn problem solve decision make understanding language recognize image analyze information artificial intelligence widely healthcare education',
 ' banking transportation business industry common example ai application include chatgpt google assistant self drive car recommendation system facial r',
 'ecognition virtual assistant artificial intelligence consist important branch include machine learn deep learn natural language processing computer vi',
 'sion technology work machine intelligent capable solve real world problem 2 machine learning ml machine learning subset artificial intelligence enable',
 ' computer learn historical datum identify pattern prediction decisio

In [12]:
chunksize = 200
overlap = 20
chunks=[data[index:index+chunksize] for index in range(0,len(data),chunksize-overlap)]
chunks

['  machine learn deep learning natural language processing nlp 1 introduction artificial intelligence artificial intelligence ai fast grow technology modern world branch computer science focus develop ',
 'ience focus develop intelligent machine computer system capable perform task normally require human intelligence task include learn problem solve decision make understanding language recognize image a',
 'ge recognize image analyze information artificial intelligence widely healthcare education banking transportation business industry common example ai application include chatgpt google assistant self ',
 'ogle assistant self drive car recommendation system facial recognition virtual assistant artificial intelligence consist important branch include machine learn deep learn natural language processing c',
 'anguage processing computer vision technology work machine intelligent capable solve real world problem 2 machine learning ml machine learning subset artificial intelligence enab

In [13]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 200,
    chunk_overlap = 25,
    separators=['\n\n','.','\n',' ','']
)
chunks = splitter.create_documents([data])

In [14]:
for chunk in range(len(chunks)):
    chunks[chunk].metadata={'file_name':'data.txt'}
chunks

[Document(metadata={'file_name': 'data.txt'}, page_content='machine learn deep learning natural language processing nlp 1 introduction artificial intelligence artificial intelligence ai fast grow technology modern world branch computer science focus develop'),
 Document(metadata={'file_name': 'data.txt'}, page_content='science focus develop intelligent machine computer system capable perform task normally require human intelligence task include learn problem solve decision make understanding language recognize'),
 Document(metadata={'file_name': 'data.txt'}, page_content='language recognize image analyze information artificial intelligence widely healthcare education banking transportation business industry common example ai application include chatgpt google'),
 Document(metadata={'file_name': 'data.txt'}, page_content='include chatgpt google assistant self drive car recommendation system facial recognition virtual assistant artificial intelligence consist important branch include mac

### 4.Embeddings & FAISS Vector Store

In [15]:
embeddings = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks,embeddings)
retriver = vectorstore.as_retriever(
                                     search_kwargs = {"k":10}
)

c:\Users\Saravanan\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6074.43it/s]


### 5.Reranking Retrieved 

In [16]:
ranker = Ranker()
def retrieve_and_rerank(query:str, top_n: int = 3):
    initial_docs = retriver.invoke(query)

    passages = [
                {"id":idx , "text":doc.page_content}
                for idx,doc in enumerate(initial_docs)
    ]

    rerank_request = RerankRequest(query=query, passages=passages)
    reranked_result = ranker.rerank(rerank_request)

    return reranked_result[:top_n]

call = retrieve_and_rerank('Explain something abt Gen AI')
print(call)

[{'id': 2, 'text': 'language recognize image analyze information artificial intelligence widely healthcare education banking transportation business industry common example ai application include chatgpt google', 'score': np.float32(0.00018629715)}, {'id': 9, 'text': 'tool support personalize learning automated assessment future technology include generative ai large language model retrieval augmented generation rag ai agents autonomous system intelligent', 'score': np.float32(8.433073e-05)}, {'id': 6, 'text': 'machine learn deep learning natural language processing nlp 1 introduction artificial intelligence artificial intelligence ai fast grow technology modern world branch computer science focus develop', 'score': np.float32(3.4448763e-05)}]


In [19]:
query =" how is deep learning used in speech recognition?"
top_chunks = retrieve_and_rerank(query,top_n=3)

for item in top_chunks:
    print(f"Score: {item['score']}")
    print(f"Text: {item['text']}")
    print("-" * 50)


Score: 0.9976219534873962
Text: real world application include facial recognition speech recognition medical image analysis autonomous vehicle image generation language translation popular framework develop deep learning model
--------------------------------------------------
Score: 0.8726791143417358
Text: linguistics computer science machine learning deep learning process large amount textual speech datum widely chatbot virtual assistant language translation sentiment analysis email filtering text
--------------------------------------------------
Score: 0.013551607728004456
Text: deep learning model include tensorflow kera pytorch 4 natural language processing nlp natural language processing branch artificial intelligence enable computer understand interpret analyze generate
--------------------------------------------------
